In [5]:
import os
import sys

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.text_utils import find_index_of_incongruent_word
from src.tokenizer_utils import load_tokenizer, tokenize

### Representation Similarity Analysis

Задача вычислить пространство представлений вычислением расстояний между стимулами в пространстве признаков

- признаковое простраство для модели - скрытые представления стимулов на каждом из слоев
- признаковое пространство для ЭЭГ - временные ряды значений потенциалов с датчиков

In [12]:
result_csv_filename = "results.csv"

non_instruct_paths = [
    "../src/results/meta-llama_Meta-Llama-3-8B/",
    "../src/results/mistralai_Mistral-7B-v0.1/",
    "../src/results/Qwen_Qwen2.5-7B/",
]

In [13]:
test_path = non_instruct_paths[1]

df_result = pd.read_csv(os.path.join(test_path, result_csv_filename))

In [14]:
df_result.head()

,sentence,model,instruct,hidden_states_path,structure,target,congruent
0,Автобусы проходят массовую дезинфекцией,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00000_mist...,Subject - Verb - Adj - Object,grammar,Автобусы проходят массовую дезинфекцию
1,Автобусы проходят массовую дезинфекцию,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00001_mist...,Subject - Verb - Adj - Object,normal,Автобусы проходят массовую дезинфекцию
2,Автобусы проходят массовую фортуной,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00002_mist...,Subject - Verb - Adj - Object,semantics_grammar,Автобусы проходят массовую дезинфекцию
3,Автобусы проходят массовую фортуну,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00003_mist...,Subject - Verb - Adj - Object,semantics,Автобусы проходят массовую дезинфекцию
4,Авторы получали подарками,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00004_mist...,Subject - Verb - Object,grammar,Авторы получали подарки


In [15]:

row_1 = df_result.iloc[0]

row_1

sentence                        Автобусы проходят массовую дезинфекцией
model                                         mistralai/Mistral-7B-v0.1
instruct                                                          False
hidden_states_path    ./results/mistralai_Mistral-7B-v0.1/00000_mist...
structure                                 Subject - Verb - Adj - Object
target                                                          grammar
congruent                        Автобусы проходят массовую дезинфекцию
Name: 0, dtype: object

In [16]:
find_index_of_incongruent_word([row_1["sentence"]], [row_1["congruent"]])


tokenizer = load_tokenizer(row_1["model"])

encoded_sentence = tokenize(
    tokenizer=tokenizer,
    sentences=[row_1["sentence"]],
    use_chat_template=row_1["instruct"],
)

2025-10-28 00:53:36 DEBUG    src.text_utils: Searching for incongruent word: дезинфекцию
2025-10-28 00:53:36 DEBUG    urllib3.connectionpool: Starting new HTTPS connection (1): huggingface.co:443
2025-10-28 00:53:36 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /mistralai/Mistral-7B-v0.1/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2025-10-28 00:53:36 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/mistralai/Mistral-7B-v0.1/27d67f1b5f57dc0953326b2601d68371d40ea8da/tokenizer_config.json HTTP/1.1" 200 0
2025-10-28 00:53:36 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "GET /api/models/mistralai/Mistral-7B-v0.1/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
2025-10-28 00:53:37 INFO     src.tokenizer_utils: Tokenizer adds prefix space


In [ ]:
for file in os.listdir(test_path):
    if ".npy" in file:
        hiddens = np.load(os.path.join(test_path, file))
        print(hiddens.shape)
        break